# Roman Urdu sentiment analysis — RNN vs LSTM vs GRU

This notebook trains three recurrent models (vanilla RNN, LSTM, GRU) on the same Roman Urdu sentiment dataset, so you can directly compare how each one handles long-range context.

**Before running:** In Colab go to `Runtime → Change runtime type → T4 GPU` to get the free GPU.

Steps: check GPU → download dataset → preprocess → build vocab → define models → train all three → compare results → save models.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — enable GPU in Runtime > Change runtime type")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Download the dataset

Roman Urdu sentiment dataset (~20,000 manually tagged sentences, Positive/Negative/Neutral), originally compiled from the UCI Roman Urdu Data Set.

In [ ]:
!wget -q -O roman_urdu.csv "https://raw.githubusercontent.com/Smat26/Roman-Urdu-Dataset/master/Dataset/Roman%20Urdu%20DataSet.csv" || \
!wget -q -O roman_urdu.csv "https://github.com/zenith-thread/Roman-Urdu-Dataset/raw/refs/heads/main/Dataset/Roman%20Urdu%20DataSet.csv"

import pandas as pd
df = pd.read_csv("roman_urdu.csv", header=None, names=["text", "sentiment", "junk"], encoding="utf-8", on_bad_lines="skip")
df = df[["text", "sentiment"]].dropna()
df["sentiment"] = df["sentiment"].str.strip()
df = df[df["sentiment"].isin(["Positive", "Negative", "Neutral"])]
print(df.shape)
print(df["sentiment"].value_counts())
df.head()

## 2. Preprocess and build a vocabulary

Roman Urdu has no standard spelling, so we keep preprocessing light: lowercase, strip punctuation/digits, split on whitespace. A more advanced version could normalize common spelling variants (e.g. "acha"/"achaa"/"acha").

In [ ]:
import re
from collections import Counter

def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["clean"] = df["text"].apply(clean_text)
df = df[df["clean"].str.len() > 0]

label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
df["label"] = df["sentiment"].map(label_map)

all_words = " ".join(df["clean"]).split()
word_counts = Counter(all_words)
vocab = ["<pad>", "<unk>"] + [w for w, c in word_counts.most_common(10000)]
word2idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
print("Vocab size:", vocab_size)

MAX_LEN = 25

def encode(text):
    ids = [word2idx.get(w, word2idx["<unk>"]) for w in text.split()][:MAX_LEN]
    ids += [word2idx["<pad>"]] * (MAX_LEN - len(ids))
    return ids

df["encoded"] = df["clean"].apply(encode)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

X = np.stack(df["encoded"].values)
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

class SentimentDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SentimentDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(SentimentDataset(X_test, y_test), batch_size=64)
print("Train size:", len(X_train), "Test size:", len(X_test))

## 3. Define the three models

Same embedding size, same hidden size, same classifier head — the only thing that changes is the recurrent cell. That controlled comparison is the point of the project.

In [ ]:
import torch.nn as nn

EMBED_DIM = 100
HIDDEN_DIM = 128
NUM_CLASSES = 3

class SentimentModel(nn.Module):
    def __init__(self, vocab_size, cell_type="RNN"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[cell_type]
        self.rnn = rnn_cls(EMBED_DIM, HIDDEN_DIM, batch_first=True)
        self.fc = nn.Linear(HIDDEN_DIM, NUM_CLASSES)
        self.cell_type = cell_type

    def forward(self, x):
        emb = self.embedding(x)
        if self.cell_type == "LSTM":
            _, (hidden, _) = self.rnn(emb)
        else:
            _, hidden = self.rnn(emb)
        return self.fc(hidden[-1])

## 4. Train each model

Same training loop, same number of epochs, same optimizer settings for all three — so any difference in results comes from the architecture, not the training setup.

In [ ]:
import torch.optim as optim
import time

def train_model(cell_type, epochs=8):
    model = SentimentModel(vocab_size, cell_type).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    history = []

    start = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        acc = evaluate(model)
        history.append((epoch + 1, avg_loss, acc))
        print(f"[{cell_type}] epoch {epoch+1}/{epochs} — loss {avg_loss:.4f} — test acc {acc:.3f}")
    print(f"[{cell_type}] trained in {time.time() - start:.1f}s")
    return model, history

def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

results = {}
histories = {}
for cell_type in ["RNN", "LSTM", "GRU"]:
    model, history = train_model(cell_type)
    results[cell_type] = model
    histories[cell_type] = history

## 5. Compare results

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
for cell_type, history in histories.items():
    epochs = [h[0] for h in history]
    accs = [h[2] for h in history]
    plt.plot(epochs, accs, marker="o", label=cell_type)
plt.xlabel("Epoch")
plt.ylabel("Test accuracy")
plt.title("RNN vs LSTM vs GRU — Roman Urdu sentiment")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("comparison_plot.png", dpi=150, bbox_inches="tight")
plt.show()

for cell_type, model in results.items():
    print(f"{cell_type}: final test accuracy = {evaluate(model):.3f}")

## 6. The "gotcha" test — a sentence that flips tone midway

This is the example that actually demonstrates the vanishing-gradient problem: a sentence that starts negative and ends positive. Vanilla RNN tends to lose the early context; LSTM/GRU should hold onto it better.

In [ ]:
def predict_all(text):
    encoded = torch.tensor([encode(clean_text(text))], dtype=torch.long).to(device)
    print(f'Input: "{text}"')
    for cell_type, model in results.items():
        model.eval()
        with torch.no_grad():
            probs = torch.softmax(model(encoded), dim=1)[0]
        pred = probs.argmax().item()
        label = {v: k for k, v in label_map.items()}[pred]
        print(f"  {cell_type:5s} -> {label:8s} ({probs[pred]*100:.1f}% confidence)")

predict_all("shuru mein bohat boring tha lekin end mein bilkul zabardast nikla")
predict_all("yeh movie bohat acha tha")
predict_all("waqt zaya hua bilkul bakwas")

## 7. Save the trained models

Download these three files and use them in your Streamlit/Gradio app for the deployed demo.

In [ ]:
for cell_type, model in results.items():
    torch.save(model.state_dict(), f"{cell_type.lower()}_sentiment.pt")

import pickle
with open("word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)

print("Saved: rnn_sentiment.pt, lstm_sentiment.pt, gru_sentiment.pt, word2idx.pkl")

from google.colab import files
for fname in ["rnn_sentiment.pt", "lstm_sentiment.pt", "gru_sentiment.pt", "word2idx.pkl"]:
    files.download(fname)

## Next steps

1. Build a small Streamlit or Gradio app that loads these three `.pt` files and reproduces the RNN vs LSTM vs GRU comparison UI.
2. Deploy it on HuggingFace Spaces (free) for a live demo link.
3. Put this notebook, the app code, a README with the accuracy comparison chart, and the demo link/GIF on GitHub.
